# Hierarchischer Ansatz: Erste Klassifikatoren

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Erst die Kultur, dann das Stadium je Kultur vorhersagen (#87) und gegen die kombinierte RBF-SVM einordnen.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.data import band_columns, load_split, load_train
from awp2.evaluation import plot_confusion_matrices
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import (
    make_combined_svm,
    make_hierarchical_extra_trees,
    make_hierarchical_extra_trees_svm,
    make_hierarchical_svm,
    oracle_crop_metrics,
)
from awp2.plots import (
    plot_class_recall,
    plot_metric_comparison,
    plot_top_confusions,
    plot_tuning_candidates,
    save_doc_figure,
)
from awp2.preprocessing import PreprocessingConfig

In [ ]:
train = load_train()
bands = band_columns(train)
train.shape

## Aufbau

`HierarchicalClassifier` (#85) trainiert ein Kultur-Modell auf allen Zeilen und je Kultur ein
Stadium-Modell auf deren Zeilen. Drei Kombinationen, jeweils per CV getunt und **einmal** auf der
Validierung bewertet:

1. RBF-SVM → RBF-SVM – wie die kombinierte Referenz, nur zweistufig
2. Extra Trees → RBF-SVM – Bäume für die Kultur, SVM für das Stadium
3. Extra Trees → Extra Trees

Alle Läufe nutzen dieselbe Vorverarbeitung: Spektren standardisiert, AEZ one-hot, Monat zyklisch
(aktueller Pipeline-Standard). Die Referenz `combined_svm_expanded` (0.866) lief noch mit
numerischem Monat; deshalb wird die kombinierte SVM mit derselben Vorverarbeitung **nur per CV**
neu eingeordnet (Abschnitt 4) – fair verglichen werden CV-Scores.

In [ ]:
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
svm_values = {"C": [3.0, 10.0, 30.0, 100.0], "gamma": [0.003, 0.01, 0.03]}
TOP_CANDIDATES = 15

## 1. RBF-SVM → RBF-SVM

Gemeinsame Suche über beide Stufen: Die Kultur-SVM und die Stadium-SVMs bekommen je eigenes `C`
und `gamma` aus dem Raster von `combined_svm_expanded_search`.

In [ ]:
svm_tuned = tune(
    make_hierarchical_svm(),
    TuneConfig(
        name="hierarchical_svm_search",
        approach="hierarchical",
        description="RBF-SVM für Kultur und je Kultur für das Stadium; C und gamma je Stufe "
        "aus dem Raster der kombinierten SVM.",
        preprocessing=preprocessing,
        param_grid={
            **{f"crop_estimator__{name}": values for name, values in svm_values.items()},
            **{f"stage_estimator__{name}": values for name, values in svm_values.items()},
        },
    ),
)
svm_tuned.candidates.sort_values("rank").head(TOP_CANDIDATES)

In [ ]:
svm_result = run(
    svm_tuned.best_model,
    RunConfig(
        name="hierarchical_svm",
        approach="hierarchical",
        description="Beste RBF-SVM → RBF-SVM aus hierarchical_svm_search, einmal bewertet.",
        preprocessing=preprocessing,
        tuning_run=svm_tuned.run_id,
    ),
)
svm_tuned.best_params, svm_tuned.best_score, svm_result.metrics.model_dump()

## 2. Extra Trees → RBF-SVM

Hypothese: Bäume trennen die Kulturen über einzelne Bänder und den Kontext robust, die SVM
trennt die feineren Stadien innerhalb einer Kultur besser.

In [ ]:
et_svm_tuned = tune(
    make_hierarchical_extra_trees_svm(),
    TuneConfig(
        name="hierarchical_et_svm_search",
        approach="hierarchical",
        description="Extra Trees für die Kultur, RBF-SVM je Kultur für das Stadium.",
        preprocessing=preprocessing,
        param_grid={
            "crop_estimator__max_features": ["sqrt", 0.3],
            **{f"stage_estimator__{name}": values for name, values in svm_values.items()},
        },
    ),
)
et_svm_tuned.candidates.sort_values("rank").head(TOP_CANDIDATES)

In [ ]:
et_svm_result = run(
    et_svm_tuned.best_model,
    RunConfig(
        name="hierarchical_et_svm",
        approach="hierarchical",
        description="Beste Extra Trees → RBF-SVM aus hierarchical_et_svm_search, einmal bewertet.",
        preprocessing=preprocessing,
        tuning_run=et_svm_tuned.run_id,
    ),
)
et_svm_tuned.best_params, et_svm_tuned.best_score, et_svm_result.metrics.model_dump()

## 3. Extra Trees → Extra Trees

Kontrollvariante ohne SVM: Profitiert auch ein Baum-Ensemble von der Zerlegung? Vergleich mit
`combined_extra_trees` (0.804).

In [ ]:
et_tuned = tune(
    make_hierarchical_extra_trees(),
    TuneConfig(
        name="hierarchical_et_search",
        approach="hierarchical",
        description="Extra Trees für Kultur und Stadium; max_features je Stufe, Blattgröße der "
        "Stadium-Bäume wegen kleiner Kulturen.",
        preprocessing=preprocessing,
        param_grid={
            "crop_estimator__max_features": ["sqrt", 0.3],
            "stage_estimator__max_features": ["sqrt", 0.3],
            "stage_estimator__min_samples_leaf": [1, 3],
        },
    ),
)
et_tuned.candidates.sort_values("rank")

In [ ]:
et_result = run(
    et_tuned.best_model,
    RunConfig(
        name="hierarchical_et",
        approach="hierarchical",
        description="Beste Extra Trees → Extra Trees aus hierarchical_et_search, einmal bewertet.",
        preprocessing=preprocessing,
        tuning_run=et_tuned.run_id,
    ),
)
et_tuned.best_params, et_tuned.best_score, et_result.metrics.model_dump()

## 4. CV-Referenz: kombinierte SVM mit derselben Vorverarbeitung

Nur CV, keine zweite Validierung der kombinierten SVM: Der Vergleich hierarchisch vs. kombiniert
läuft über den CV-Score bei identischer Vorverarbeitung und identischem Raster.

In [ ]:
combined_tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_cv_reference",
        approach="combined",
        description="CV-only: kombinierte RBF-SVM mit der Vorverarbeitung der hierarchischen "
        "Läufe (standardisierte Spektren, zyklischer Monat) als fairer Vergleich.",
        preprocessing=preprocessing,
        param_grid={f"estimator__{name}": values for name, values in svm_values.items()},
    ),
)
combined_tuned.candidates.sort_values("rank").head(5)

In [ ]:
cv_scores = pd.DataFrame(
    {
        "CV BAcc kombiniert": {
            "Kombiniert: RBF-SVM": combined_tuned.best_score,
            "Hierarchisch: RBF-SVM → RBF-SVM": svm_tuned.best_score,
            "Hierarchisch: Extra Trees → RBF-SVM": et_svm_tuned.best_score,
            "Hierarchisch: Extra Trees → Extra Trees": et_tuned.best_score,
        },
        "Validierung BAcc kombiniert": {
            "Kombiniert: RBF-SVM": np.nan,
            "Hierarchisch: RBF-SVM → RBF-SVM": svm_result.metrics.bacc_combined,
            "Hierarchisch: Extra Trees → RBF-SVM": et_svm_result.metrics.bacc_combined,
            "Hierarchisch: Extra Trees → Extra Trees": et_result.metrics.bacc_combined,
        },
    }
)
cv_scores.round(3)

## 5. Vergleich auf der Validierung

In [ ]:
results = {
    "RBF-SVM → RBF-SVM": svm_result,
    "Extra Trees → RBF-SVM": et_svm_result,
    "Extra Trees → Extra Trees": et_result,
}
comparison_ax = plot_metric_comparison({label: r.metrics for label, r in results.items()})
comparison_ax.set_title("Hierarchisch: Klassifikatoren im Vergleich");

## 6. Fehlerfortpflanzung: Oracle-Kultur

Wie gut wäre das Stadium, wenn die Kultur immer stimmt? Dafür bekommen die Stadium-Modelle die
**wahre** Kultur (`predict_stage`). Der Abstand zur echten Vorhersage ist der Anteil, den
Kulturfehler am Stadium kosten. Die vollständige Zerlegung folgt in #32.

In [ ]:
split = load_split()
oracle_results = {
    label: oracle_crop_metrics(r.pipeline, split.X_val, split.y_val) for label, r in results.items()
}
oracle = pd.DataFrame(
    {
        label: {
            "BAcc Stadium": r.metrics.bacc_stage,
            "BAcc Stadium (Oracle-Kultur)": oracle_results[label].bacc_stage,
            "BAcc kombiniert": r.metrics.bacc_combined,
            "BAcc kombiniert (Oracle-Kultur)": oracle_results[label].bacc_combined,
        }
        for label, r in results.items()
    }
).T
oracle.round(3)

## 7. Bestes hierarchisches Modell im Detail

Gewählt nach dem CV-Score.

In [ ]:
best_label, best_tuned, best_result = max(
    [
        ("RBF-SVM → RBF-SVM", svm_tuned, svm_result),
        ("Extra Trees → RBF-SVM", et_svm_tuned, et_svm_result),
        ("Extra Trees → Extra Trees", et_tuned, et_result),
    ],
    key=lambda entry: entry[1].best_score,
)
best_label

In [ ]:
tuning_ax = plot_tuning_candidates(
    best_tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"),
    validation_score=best_result.metrics.bacc_combined,
)
tuning_ax.set_title(f"Hierarchisch {best_label}: beste {TOP_CANDIDATES} CV-Kandidaten");

In [ ]:
confusion_figure = plot_confusion_matrices(best_result.y_val, best_result.y_pred)
recall_figure = plot_class_recall(best_result.y_val, best_result.y_pred)
confusions_figure = plot_top_confusions(best_result.y_val, best_result.y_pred)

In [ ]:
save_doc_figure(comparison_ax.figure, "hierarchical_comparison", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(tuning_ax.figure, "hierarchical_tuning", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(confusion_figure, "hierarchical_confusion", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(recall_figure, "hierarchical_recall", MODEL_DOCS_FIGURES_DIR)
save_doc_figure(confusions_figure, "hierarchical_confusions", MODEL_DOCS_FIGURES_DIR)

In [ ]:
runs = pd.DataFrame(
    {
        label: {
            "tuning_run": tuned.run_id,
            "evaluation_run": result.run_id if result else None,
            "best_cv_score": round(tuned.best_score, 4),
            "best_params": tuned.best_params,
        }
        for label, tuned, result in [
            ("hierarchical_svm", svm_tuned, svm_result),
            ("hierarchical_et_svm", et_svm_tuned, et_svm_result),
            ("hierarchical_et", et_tuned, et_result),
            ("combined_svm_cv_reference", combined_tuned, None),
        ]
    }
).T
runs